# Functions & Libraries

In [0]:
#Call libraries
from pyspark.sql import functions as F
import pandas as pd
import inspect
import random

## Baseline functions

In [0]:
#Funtion to return valid candidates for an empty Sudoku cell
def get_cell_candidates(grid, row, col):
    """ Return valid candidates for an empty Sudoku cell. """

    if grid[row][col] != 0:
        return []

    row_values = set(grid[row])
    column_values = {grid[r][col] for r in range(9)}

    box_row = (row // 3) * 3
    box_col = (col // 3) * 3

    box_values = {
        grid[r][c]
        for r in range(box_row, box_row + 3)
        for c in range(box_col, box_col + 3)
    }

    used_numbers = row_values | column_values | box_values

    return sorted(set(range(1, 10)) - used_numbers)

#Funtion to apply the Naked Single technique
def solve_naked_singles(grid):
    """ Apply Naked Single moves until no further progress is possible. """

    board = [row.copy() for row in grid]
    naked_single_steps = 0

    while True:
        progress = False

        for row in range(9):
            for col in range(9):
                if board[row][col] == 0:
                    candidates = get_cell_candidates(board, row, col)

                    if len(candidates) == 1: #If there is exactly one candidate, it fills the cell and counts the operation
                        board[row][col] = candidates[0]
                        naked_single_steps += 1
                        progress = True

        if not progress:
            break

    solved = all(
        board[row][col] != 0
        for row in range(9)
        for col in range(9)
    )

    return {
        "board": board,
        "naked_single_steps": naked_single_steps,
        "solved": solved,
        "remaining_empty_cells": sum(
            cell == 0
            for row in board
            for cell in row
        )
    }

    #Notes:
        #If solved = True, it means the Sudoku has been completely solved using this technique
        #If solved = False, it means we will need other strategies to continue the solution process

#function to apply hidden single technique
def find_hidden_singles(grid):
    """ Find cells that contain the only possible position for a number within a row, column, or 3x3 box. """

    hidden_singles = {}

    units = []

    # Rows
    for row in range(9):
        units.append(
            [(row, col) for col in range(9)]
        )

    # Columns
    for col in range(9):
        units.append(
            [(row, col) for row in range(9)]
        )

    # 3x3 boxes
    for box_row in range(0, 9, 3):
        for box_col in range(0, 9, 3):
            units.append([
                (row, col)
                for row in range(box_row, box_row + 3)
                for col in range(box_col, box_col + 3)
            ])

    for unit in units:
        for number in range(1, 10):
            possible_cells = []

            for row, col in unit:
                if grid[row][col] == 0:
                    candidates = get_cell_candidates(grid, row, col)

                    if number in candidates:
                        possible_cells.append((row, col))

            if len(possible_cells) == 1:
                row, col = possible_cells[0]

                if (row, col) not in hidden_singles:
                    hidden_singles[(row, col)] = number

    return hidden_singles

#Funtion to apply the Hidden Single technique with the Naked Single technique
def solve_with_singles(grid):
    """ Solve a Sudoku using Naked Singles and Hidden Singles. """

    board = [row.copy() for row in grid]

    naked_single_steps = 0
    hidden_single_steps = 0

    while True:
        progress = False

        #Apply Naked Singles first
        for row in range(9):
            for col in range(9):
                if board[row][col] == 0:
                    candidates = get_cell_candidates(board, row, col)

                    if len(candidates) == 1:
                        board[row][col] = candidates[0]
                        naked_single_steps += 1
                        progress = True

        if progress:
            continue

        #Apply one Hidden Single, then recalculate all candidates
        hidden_singles = find_hidden_singles(board)

        for (row, col), number in hidden_singles.items():
            if board[row][col] == 0:
                candidates = get_cell_candidates(board, row, col)

                if number in candidates:
                    board[row][col] = number
                    hidden_single_steps += 1
                    progress = True
                    break

        if not progress:
            break

    remaining_empty_cells = sum(
        cell == 0
        for row in board
        for cell in row
    )

    return {
        "board": board,
        "naked_single_steps": naked_single_steps,
        "hidden_single_steps": hidden_single_steps,
        "solved": remaining_empty_cells == 0,
        "remaining_empty_cells": remaining_empty_cells,
    }

#Funtion to apply the Naked Single and Hidden Single techniques with the number of rounds 
def solve_with_singles_and_rounds(grid):
    """ Solve a Sudoku and track logical solving rounds. """

    board = [row.copy() for row in grid]

    naked_single_steps = 0
    hidden_single_steps = 0
    solver_rounds = 0

    while True:
        round_progress = False

        #Apply Naked Singles until no more are available
        while True:
            naked_progress = False

            for row in range(9):
                for col in range(9):
                    if board[row][col] == 0:
                        candidates = get_cell_candidates(board, row, col)

                        if len(candidates) == 1:
                            board[row][col] = candidates[0]
                            naked_single_steps += 1
                            naked_progress = True
                            round_progress = True

            if not naked_progress:
                break

        #Apply one Hidden Single at a time
        hidden_singles = find_hidden_singles(board)

        for (row, col), number in hidden_singles.items():
            if board[row][col] == 0:
                candidates = get_cell_candidates(board, row, col)

                if number in candidates:
                    board[row][col] = number
                    hidden_single_steps += 1
                    round_progress = True
                    break

        if not round_progress:
            break

        solver_rounds += 1

    remaining_empty_cells = sum(
        cell == 0
        for row in board
        for cell in row
    )

    return {
        "board": board,
        "naked_single_steps": naked_single_steps,
        "hidden_single_steps": hidden_single_steps,
        "solver_rounds": solver_rounds,
        "solved": remaining_empty_cells == 0,
        "remaining_empty_cells": remaining_empty_cells,
    }

#Function to apply the Backtracking and MRV techniques
def solve_with_backtracking(grid):
    """ Solve a Sudoku using deterministic backtracking and MRV. """

    board = [row.copy() for row in grid]

    search_nodes = 0
    backtracks = 0
    max_search_depth = 0

    def search(depth=0):
        nonlocal search_nodes, backtracks, max_search_depth

        search_nodes += 1
        max_search_depth = max(max_search_depth, depth)

        best_cell = None
        best_candidates = None

        # Select the empty cell with the fewest valid candidates.
        for row in range(9):
            for col in range(9):
                if board[row][col] != 0:
                    continue

                candidates = get_cell_candidates(board, row, col)

                if not candidates:
                    backtracks += 1
                    return False

                if (
                    best_candidates is None
                    or len(candidates) < len(best_candidates)
                ):
                    best_cell = (row, col)
                    best_candidates = candidates

                    if len(best_candidates) == 1:
                        break

            if best_candidates is not None and len(best_candidates) == 1:
                break

        # No empty cells remain.
        if best_cell is None:
            return True

        row, col = best_cell

        for number in best_candidates:
            board[row][col] = number

            if search(depth + 1):
                return True

            board[row][col] = 0
            backtracks += 1

        return False

    solved = search()

    return {
        "board": board,
        "solved": solved,
        "search_nodes": search_nodes,
        "backtracks": backtracks,
        "max_search_depth": max_search_depth,
    }

##Advanced functions

In [0]:
#Funtion to find naked pairs (==2) and return candidate eliminations by cell
def find_naked_pairs(grid):
    """ Find naked pairs and return candidate eliminations by cell. """

    eliminations = {}
    units = []

    #Rows
    for row in range(9):
        units.append([
            (row, col) for col in range(9)
        ])

    #Columns
    for col in range(9):
        units.append([
            (row, col) for row in range(9)
        ])

    #3x3 boxes
    for box_row in range(0, 9, 3):
        for box_col in range(0, 9, 3):
            units.append([
                (row, col)
                for row in range(box_row, box_row + 3)
                for col in range(box_col, box_col + 3)
            ])

    for unit in units:
        candidate_cells = {}

        #Group empty cells by their exact candidate pairs
        for row, col in unit:
            candidates = get_cell_candidates(grid, row, col)

            if len(candidates) == 2:
                pair = tuple(candidates)
                candidate_cells.setdefault(pair, []).append(
                    (row, col)
                )

        #A pair is valid when exactly two cells share it
        for pair, cells in candidate_cells.items():
            if len(cells) != 2:
                continue

            pair_numbers = set(pair)

            for row, col in unit:
                if (row, col) in cells or grid[row][col] != 0:
                    continue

                candidates = get_cell_candidates(grid, row, col)
                removable = set(candidates) & pair_numbers

                if removable:
                    eliminations.setdefault((row, col), set()).update(
                        removable
                    )

    return eliminations

#Function to combine Naked Singles and Hidden Singles with Naked Pair techniques
def solve_with_singles_and_pairs(grid):
    """Solve a Sudoku using Naked Singles, Hidden Singles, and Naked Pairs."""

    naked_single_steps = 0
    hidden_single_steps = 0
    naked_pair_eliminations = 0

    while True:
        progress = False

        #Apply Naked Singles
        for row in range(9):
            for col in range(9):
                if grid[row][col] != 0:
                    continue

                candidates = get_cell_candidates(grid, row, col)

                if len(candidates) == 1:
                    grid[row][col] = candidates[0]
                    naked_single_steps += 1
                    progress = True

        if progress:
            continue

        #Apply one Hidden Single at a time
        hidden_singles = find_hidden_singles(grid)

        if hidden_singles:
            (row, col), number = next(iter(hidden_singles.items()))
            grid[row][col] = number
            hidden_single_steps += 1
            continue

        #Apply Naked Pair eliminations
        eliminations = find_naked_pairs(grid)

        if eliminations:
            #Candidate eliminations are counted for analysis
            #The current candidate function recalculates candidates from the grid, so eliminations are not persisted between iterations
            naked_pair_eliminations += sum(
                len(numbers) for numbers in eliminations.values()
            )

        #No values were filled and no persistent candidate state exists
        #Stop rather than repeating the same pair eliminations indefinitely
        break

    solved = all(
        grid[row][col] != 0
        for row in range(9)
        for col in range(9)
    )

    remaining_empty_cells = sum(
        1
        for row in range(9)
        for col in range(9)
        if grid[row][col] == 0
    )

    return {
        "grid": grid,
        "naked_single_steps": naked_single_steps,
        "hidden_single_steps": hidden_single_steps,
        "naked_pair_eliminations": naked_pair_eliminations,
        "solved": solved,
        "remaining_empty_cells": remaining_empty_cells,
    }

#Function to create a candidate map
def initialize_candidates(grid):
    """ Build a persistent candidate set for every empty cell. """

    candidates = {}

    for row in range(9):
        for col in range(9):
            if grid[row][col] == 0:
                candidates[(row, col)] = set(
                    get_cell_candidates(grid, row, col)
                )

    return candidates

#Function to elimate candidates based on Naked Pairs
def apply_naked_pairs(grid, candidates):
    """ Eliminate naked pair candidates within rows, columns, and boxes. """

    eliminations_count = 0
    units = []

    #Rows
    for row in range(9):
        units.append([(row, col) for col in range(9)])

    #Columns
    for col in range(9):
        units.append([(row, col) for row in range(9)])

    #3x3 boxes
    for box_row in range(0, 9, 3):
        for box_col in range(0, 9, 3):
            units.append([
                (row, col)
                for row in range(box_row, box_row + 3)
                for col in range(box_col, box_col + 3)
            ])

    for unit in units:
        candidate_cells = {}

        #Group cells that have exactly two candidates
        for cell in unit:
            if cell not in candidates:
                continue

            cell_candidates = candidates[cell]

            if len(cell_candidates) == 2:
                pair = tuple(sorted(cell_candidates))
                candidate_cells.setdefault(pair, []).append(cell)

        #A naked pair must occur in exactly two cells
        for pair, pair_cells in candidate_cells.items():
            if len(pair_cells) != 2:
                continue

            pair_numbers = set(pair)

            #Remove the pair numbers from other cells in the unit
            for cell in unit:
                if cell in pair_cells or cell not in candidates:
                    continue

                removable = candidates[cell] & pair_numbers

                if removable:
                    candidates[cell] -= removable
                    eliminations_count += len(removable)

    return eliminations_count

#Funtion to propagate the sudoku puzzle candidates
def propagate_sudoku(grid, candidates):
    """ Apply singles and naked pairs until no further progress is possible. """

    naked_single_steps = 0
    hidden_single_steps = 0
    naked_pair_eliminations = 0

    while True:
        progress = False

        #Detect contradictions
        if any(len(values) == 0 for values in candidates.values()):
            return {
                "solved": False,
                "contradiction": True,
                "naked_single_steps": naked_single_steps,
                "hidden_single_steps": hidden_single_steps,
                "naked_pair_eliminations": naked_pair_eliminations,
            }

        #Apply Naked Singles
        singles = [
            (cell, next(iter(values)))
            for cell, values in candidates.items()
            if len(values) == 1
        ]

        if singles:
            for (row, col), number in singles:
                grid[row][col] = number

            naked_single_steps += len(singles)

            #Rebuild candidates after filling cells
            candidates.clear()
            candidates.update(initialize_candidates(grid))
            progress = True
            continue

        #Apply one Hidden Single at a time
        hidden_singles = find_hidden_singles(grid)

        if hidden_singles:
            (row, col), number = next(iter(hidden_singles.items()))
            grid[row][col] = number
            hidden_single_steps += 1

            candidates.clear()
            candidates.update(initialize_candidates(grid))
            continue

        #Apply Naked Pair eliminations
        eliminated = apply_naked_pairs(grid, candidates)

        if eliminated:
            naked_pair_eliminations += eliminated
            progress = True

        if not progress:
            break

    solved = all(
        grid[row][col] != 0
        for row in range(9)
        for col in range(9)
    )

    return {
        "solved": solved,
        "contradiction": False,
        "naked_single_steps": naked_single_steps,
        "hidden_single_steps": hidden_single_steps,
        "naked_pair_eliminations": naked_pair_eliminations,
    }

#New function to create a new candidate map
def new_initialize_candidates(grid):
    """ Build a persistent candidate set for every empty cell. """

    candidates = {}

    for row in range(9):
        for col in range(9):
            if grid[row][col] == 0:
                candidates[(row, col)] = set(
                    get_cell_candidates(grid, row, col)
                )

    return candidates

#New function to propagate the sudoku puzzle candidates
def new_propagate_function(grid, candidates):
    """ 
        Apply Naked Pairs, Naked Singles, and Hidden Singles until stable. 
        The sequence will be:
            1. Check for contradictions.
            2. Apply Naked Pairs and account for any eliminations.
            3. Fill in a Naked Single, if one exists.
            4. Otherwise, fill in a Hidden Single.
            5. Repeat until the Sudoku is solved or no further progress can be made.
    """

    naked_single_steps = 0
    hidden_single_steps = 0
    naked_pair_eliminations = 0

    while True:
        # Stop if any cell has no candidates.
        if any(not values for values in candidates.values()):
            return {
                "solved": False,
                "contradiction": True,
                "naked_single_steps": naked_single_steps,
                "hidden_single_steps": hidden_single_steps,
                "naked_pair_eliminations": naked_pair_eliminations,
            }

        #Apply Naked Pair eliminations first
        eliminated = apply_naked_pairs(grid, candidates)
        naked_pair_eliminations += eliminated

        #Check for contradictions after candidate elimination
        if any(not values for values in candidates.values()):
            return {
                "solved": False,
                "contradiction": True,
                "naked_single_steps": naked_single_steps,
                "hidden_single_steps": hidden_single_steps,
                "naked_pair_eliminations": naked_pair_eliminations,
            }

        #Apply one Naked Single at a time
        naked_singles = [
            (cell, next(iter(values)))
            for cell, values in candidates.items()
            if len(values) == 1
        ]

        if naked_singles:
            (row, col), number = naked_singles[0]
            grid[row][col] = number
            naked_single_steps += 1
            del candidates[(row, col)]

            peers = set()
            peers.update((row, c) for c in range(9) if c != col)
            peers.update((r, col) for r in range(9) if r != row)

            box_row = (row // 3) * 3
            box_col = (col // 3) * 3

            peers.update(
                (r, c)
                for r in range(box_row, box_row + 3)
                for c in range(box_col, box_col + 3)
                if (r, c) != (row, col)
            )

            for peer in peers:
                if peer in candidates:
                    candidates[peer].discard(number)

            continue

        # Apply one Hidden Single at a time.
        hidden_singles = new_find_hidden_singles(grid, candidates)

        if hidden_singles:
            (row, col), number = next(iter(hidden_singles.items()))
            grid[row][col] = number
            hidden_single_steps += 1
            del candidates[(row, col)]

            peers = set()
            peers.update((row, c) for c in range(9) if c != col)
            peers.update((r, col) for r in range(9) if r != row)

            box_row = (row // 3) * 3
            box_col = (col // 3) * 3

            peers.update(
                (r, c)
                for r in range(box_row, box_row + 3)
                for c in range(box_col, box_col + 3)
                if (r, c) != (row, col)
            )

            for peer in peers:
                if peer in candidates:
                    candidates[peer].discard(number)

            continue

        # No further solving technique can make progress.
        break

    solved = (
        not candidates
        and all(
            grid[row][col] != 0
            for row in range(9)
            for col in range(9)
        )
    )

    return {
        "solved": solved,
        "contradiction": False,
        "naked_single_steps": naked_single_steps,
        "hidden_single_steps": hidden_single_steps,
        "naked_pair_eliminations": naked_pair_eliminations,
    }

#Funtion to find Hidden Singles using the persistent candidate map
def new_find_hidden_singles(grid, candidates):
    """ Find Hidden Singles using the persistent candidate map. """

    hidden_singles = {}
    units = []

    #Rows
    for row in range(9):
        units.append([(row, col) for col in range(9)])

    #Columns
    for col in range(9):
        units.append([(row, col) for row in range(9)])

    #3x3 boxes
    for box_row in range(0, 9, 3):
        for box_col in range(0, 9, 3):
            units.append([
                (row, col)
                for row in range(box_row, box_row + 3)
                for col in range(box_col, box_col + 3)
            ])

    for unit in units:
        present_numbers = {
            grid[row][col]
            for row, col in unit
            if grid[row][col] != 0
        }

        missing_numbers = set(range(1, 10)) - present_numbers

        for number in missing_numbers:
            possible_cells = [
                cell
                for cell in unit
                if cell in candidates and number in candidates[cell]
            ]

            if len(possible_cells) == 1:
                cell = possible_cells[0]

                #Avoid conflicting assignments for the same cell
                if cell not in hidden_singles:
                    hidden_singles[cell] = number
                elif hidden_singles[cell] != number:
                    hidden_singles.pop(cell)

    return hidden_singles


# Problem-solving technique

## Baseline metrics

In [0]:
#Baseline solvers:
    #naked_single_steps -> Entries made using the "Naked Single" technique -> A cell where only one candidate is possible
    #hidden_single_steps -> Entries made using the "Hidden Single" technique -> A number that can only occupy one position within a given unit
    #backtracking_nodes_MRV -> The number of nodes explored during the search with MRV -> Exploratory attempts and subsequent backtracks, used when logical techniques are insufficient to solve the puzzle

### Naked Single 

In [0]:
#Naked Single technique test one case
gold_df = spark.table("workspace.sudoku_gold.game_features")
sample_puzzle = gold_df.select("quizzes").first()["quizzes"]

grid = [
    [int(sample_puzzle[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

result = solve_naked_singles(grid)

print(f"Naked Single steps: {result['naked_single_steps']}") 
print(f"Remaining empty cells: {result['remaining_empty_cells']}")
print(f"Solved: {result['solved']}")


In [0]:
#Retrieve the reference solution for the same puzzle
sample_record = (
    spark.table("workspace.sudoku_bronze.raw_games")
    .filter(F.col("quizzes") == sample_puzzle)
    .select("solutions")
    .first()
)

reference_solution = sample_record["solutions"]

#Convert the reference solution into a 9x9 grid
reference_grid = [
    [int(reference_solution[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

#Compare the solver output with the reference solution
solution_matches = result["board"] == reference_grid

print(f"Solver completed: {result['solved']}")
print(f"Solution matches reference: {solution_matches}")
print(f"Naked Single steps: {result['naked_single_steps']}")

In [0]:
#Naked Single technique test one hundred case
sample_df = (
    spark.table("workspace.sudoku_bronze.raw_games")
    .select("quizzes", "solutions")
    .limit(100)
    .collect()
)

test_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_naked_singles(grid)

    test_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "remaining_empty_cells": result["remaining_empty_cells"],
    })

total_puzzles = len(test_results)
solved_count = sum(result["solved"] for result in test_results)
matching_count = sum(result["solution_matches"] for result in test_results)

print(f"Puzzles tested: {total_puzzles}")
print(f"Solved by Naked Singles: {solved_count}")
print(f"Solutions matching reference: {matching_count}")
print(f"Solve rate: {solved_count / total_puzzles:.2%}")

In [0]:
#Naked Single technique test one thousand random case
sample_df = (
    spark.table("workspace.sudoku_bronze.raw_games")
    .select("quizzes", "solutions")
    .orderBy(F.rand(seed=42))
    .limit(1000)
    .collect()
)

test_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_naked_singles(grid)

    test_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "remaining_empty_cells": result["remaining_empty_cells"],
    })

total_puzzles = len(test_results)
solved_count = sum(item["solved"] for item in test_results)
matching_count = sum(item["solution_matches"] for item in test_results)

print(f"Puzzles tested: {total_puzzles}")
print(f"Solved by Naked Singles: {solved_count}")
print(f"Solutions matching reference: {matching_count}")
print(f"Solve rate: {solved_count / total_puzzles:.2%}")

In [0]:
#Analyze the number of candidates and the solved cells
results_df = pd.DataFrame(test_results)

print("Naked Single steps — descriptive statistics")
print(results_df["naked_single_steps"].describe())

print("\nNaked Single steps — frequency distribution")
print(
    results_df["naked_single_steps"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nRemaining empty cells — frequency distribution")
print(
    results_df["remaining_empty_cells"]
    .value_counts()
    .sort_index()
    .to_string()
)

In [0]:
#Notes:
    #The solver is working, but we need to be careful with the interpretation
    #The puzzles contain between 45 and 50 empty cells that are solved by the algorithm
    #Since all of them were solved, the `naked_single_steps` count matches the number of empty cells in these cases
    #Therefore, this variable essentially replicates information we already have in the Gold layer: `empty_cells = 81 - given_cells`

In [0]:
#Naked Single technique test ten thousand case
sample_df = (
    spark.table("workspace.sudoku_bronze.raw_games")
    .select("quizzes", "solutions")
    .sample(withReplacement=False, fraction=0.02, seed=42)
    .limit(10_000)
    .collect()
)

test_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_naked_singles(grid)

    test_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "remaining_empty_cells": result["remaining_empty_cells"],
        "quizzes": puzzle,
        "reference_solution": reference_solution
    })

total_puzzles = len(test_results)
solved_count = sum(item["solved"] for item in test_results)
matching_count = sum(item["solution_matches"] for item in test_results)

print(f"Puzzles tested: {total_puzzles}")
print(f"Solved by Naked Singles: {solved_count}")
print(f"Solutions matching reference: {matching_count}")
print(f"Solve rate: {solved_count / total_puzzles:.2%}")

In [0]:
#Investigate the 4 unsolved puzzles
unsolved_results = [
    item for item in test_results
    if not item["solved"]
]

print(f"Unsolved puzzles: {len(unsolved_results)}")

for index, item in enumerate(unsolved_results, start=1):
    print(
        f"Puzzle {index}: "
        f"Naked Single steps = {item['naked_single_steps']}, "
        f"Remaining empty cells = {item['remaining_empty_cells']}"
    )

### Hidden Single 

In [0]:
#Discover the cells not solved by Naked Singles
first_unsolved = next(
    item for item in test_results
    if not item["solved"]
)

puzzle = first_unsolved["quizzes"]

grid = [
    [int(puzzle[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

hidden_singles = find_hidden_singles(grid)

print(f"Hidden Singles found: {len(hidden_singles)}")
print(hidden_singles)

### Hidden Single + Naked Single

In [0]:
#Test the combined solver on the puzzle that got stuck
result = solve_with_singles(grid)

print(f"Solved: {result['solved']}")
print(f"Naked Single steps: {result['naked_single_steps']}")
print(f"Hidden Single steps: {result['hidden_single_steps']}")
print(f"Remaining empty cells: {result['remaining_empty_cells']}")

In [0]:
#Test the one thousand puzzles using combined solver
test_results_combined = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_with_singles(grid)

    test_results_combined.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "hidden_single_steps": result["hidden_single_steps"],
        "remaining_empty_cells": result["remaining_empty_cells"],
    })

total_puzzles = len(test_results_combined)
solved_count = sum(item["solved"] for item in test_results_combined)
matching_count = sum(
    item["solution_matches"] for item in test_results_combined
)

print(f"Puzzles tested: {total_puzzles}")
print(f"Solved by combined strategies: {solved_count}")
print(f"Solutions matching reference: {matching_count}")
print(f"Solve rate: {solved_count / total_puzzles:.2%}")

In [0]:
#Analyze how many entries relied on each strategy and how many puzzles required Hidden Singles.
combined_df = pd.DataFrame(test_results_combined)

print("Naked Single steps")
print(combined_df["naked_single_steps"].describe())

print("\nHidden Single steps")
print(combined_df["hidden_single_steps"].describe())

print("\nPuzzles requiring Hidden Singles")
print(
    (combined_df["hidden_single_steps"] > 0).sum()
)

print("\nHidden Single steps distribution")
print(
    combined_df["hidden_single_steps"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nSolution validation")
print(
    f"All solved: {combined_df['solved'].all()}"
)
print(
    f"All solutions match reference: "
    f"{combined_df['solution_matches'].all()}"
)

In [0]:
#Notes:
    #The results reveal something very important about the dataset: Hidden Singles were required in only 4 out of the 10,000 puzzles (0.04%). The other 9,996 were solved exclusively using Naked Singles

#Insights:
    #The Hidden Singles strategy is not a common technique for solving Sudoku puzzles. It is a more advanced technique that is typically used when other strategies have been exhausted
    #Measure the resolution rounds in the combined solver


In [0]:
#Test the combined solver with round counter
record = sample_df[0]
puzzle = record["quizzes"]

grid = [
    [int(puzzle[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

result = solve_with_singles_and_rounds(grid)

print(f"Solved: {result['solved']}")
print(f"Naked Single steps: {result['naked_single_steps']}")
print(f"Hidden Single steps: {result['hidden_single_steps']}")
print(f"Solver rounds: {result['solver_rounds']}")
print(f"Remaining empty cells: {result['remaining_empty_cells']}")


In [0]:

#Run the solver on the full sample
round_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_with_singles_and_rounds(grid)

    round_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "naked_single_steps": result["naked_single_steps"],
        "hidden_single_steps": result["hidden_single_steps"],
        "solver_rounds": result["solver_rounds"],
        "remaining_empty_cells": result["remaining_empty_cells"],
    })

round_df = pd.DataFrame(round_results)

print("Solver rounds — descriptive statistics")
print(round_df["solver_rounds"].describe())

print("\nSolver rounds — frequency distribution")
print(
    round_df["solver_rounds"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nValidation")
print(f"All solved: {round_df['solved'].all()}")
print(f"All solutions match reference: {round_df['solution_matches'].all()}")


In [0]:
#Evaluate the diversity of the Gold features
sample_puzzles = [record["quizzes"] for record in sample_df]

gold_sample_df = (
    spark.table("workspace.sudoku_gold.game_features")
    .filter(F.col("quizzes").isin(sample_puzzles))
    .select(
        "given_cells",
        "row_clue_std",
        "column_clue_std",
        "box_clue_std",
        "avg_candidate_count",
        "single_candidate_cells",
        "max_candidate_count",
    )
)

gold_sample_pd = gold_sample_df.toPandas()

print("Gold feature summary")
print(gold_sample_pd.describe().round(4))

print("\nGiven cells distribution")
print(
    gold_sample_pd["given_cells"]
    .value_counts()
    .sort_index()
    .to_string()
)

In [0]:
#Notes:
    #The distribution of `given_cells` is highly concentrated: 4,465 puzzles have 34 numbers filled in at the start, and 9,181 have between 33 and 35. This helps explain why the solver's behavior is so uniform across this sample, though it does not yet prove that the number of clues is the sole cause

    #We also observe variation in the "Gold" features—specifically, `avg_candidate_count`, `single_candidate_cells`, and the standard deviations of the clues vary from one puzzle to another. Therefore, while we have structural data to investigate, we still require a target variable that represents something beyond these features themselves


In [0]:
#Insights:
    #Generate an algorithmic complexity target
    #Implement a solver using deterministic search, selecting first the empty cell with the fewest candidates (MRV — Minimum Remaining Values)
    #Count `search_nodes`, `backtracks`, and `max_search_depth`
    #Validate the solutions against the dataset's ground truth
    #Analyze the distribution of these metrics across the same 10,000 puzzles
    #Only then, decide whether they can serve as a target for regression and how to define difficulty classes

### Backtracking and MRV

In [0]:
#Test the backtracking solver
record = sample_df[0]
puzzle = record["quizzes"]
reference_solution = record["solutions"]

grid = [
    [int(puzzle[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

result = solve_with_backtracking(grid)

reference_grid = [
    [
        int(reference_solution[row * 9 + col])
        for col in range(9)
    ]
    for row in range(9)
]

print(f"Solved: {result['solved']}")
print(f"Solution matches reference: {result['board'] == reference_grid}")
print(f"Search nodes: {result['search_nodes']}")
print(f"Backtracks: {result['backtracks']}")
print(f"Maximum search depth: {result['max_search_depth']}")

In [0]:
#Notes:
    #Even though backtracking was used, this puzzle was solved without exploring alternatives

In [0]:
#Check whether the metrics vary across the 10,000 puzzles
backtracking_results = []

for record in sample_df:
    puzzle = record["quizzes"]
    reference_solution = record["solutions"]

    grid = [
        [int(puzzle[row * 9 + col]) for col in range(9)]
        for row in range(9)
    ]

    reference_grid = [
        [
            int(reference_solution[row * 9 + col])
            for col in range(9)
        ]
        for row in range(9)
    ]

    result = solve_with_backtracking(grid)

    backtracking_results.append({
        "solved": result["solved"],
        "solution_matches": result["board"] == reference_grid,
        "search_nodes": result["search_nodes"],
        "backtracks": result["backtracks"],
        "max_search_depth": result["max_search_depth"],
    })

backtracking_df = pd.DataFrame(backtracking_results)

print("Search nodes — descriptive statistics")
print(backtracking_df["search_nodes"].describe())

print("\nBacktracks — descriptive statistics")
print(backtracking_df["backtracks"].describe())

print("\nMaximum search depth — descriptive statistics")
print(backtracking_df["max_search_depth"].describe())

print("\nValidation")
print(f"All solved: {backtracking_df['solved'].all()}")
print(
    "All solutions match reference: "
    f"{backtracking_df['solution_matches'].all()}"
)

print("\nBacktracks frequency")
print(
    backtracking_df["backtracks"]
    .value_counts()
    .sort_index()
    .to_string()
)

In [0]:
#Notes:
    #9,998 puzzles required no backtracking
    #Only 2 required backtracking: one with 5 backtracks and the other with 15 backtracks
    #The number of `search_nodes` ranged from 46 to 63, though this variation is strongly linked to the number of cells filled during the search
    #The maximum depth also closely tracks the number of empty cells

In [0]:
#Identify puzzles that required backtracking
exceptional_indices = backtracking_df.index[
    backtracking_df["backtracks"] > 0
].tolist()

print(f"Puzzles requiring backtracking: {len(exceptional_indices)}")

exceptional_puzzles = []

for index in exceptional_indices:
    record = sample_df[index]
    metrics = backtracking_df.loc[index]

    exceptional_puzzles.append({
        "sample_index": index,
        "quizzes": record["quizzes"],
        "solutions": record["solutions"],
        "search_nodes": int(metrics["search_nodes"]),
        "backtracks": int(metrics["backtracks"]),
        "max_search_depth": int(metrics["max_search_depth"]),
    })

exceptional_quizzes = [
    item["quizzes"] for item in exceptional_puzzles
]

gold_exceptional_df = (
    spark.table("workspace.sudoku_gold.game_features")
    .filter(F.col("quizzes").isin(exceptional_quizzes))
    .select(
        "quizzes",
        "given_cells",
        "row_clue_std",
        "column_clue_std",
        "box_clue_std",
        "avg_candidate_count",
        "single_candidate_cells",
        "max_candidate_count",
    )
    .toPandas()
)

exceptional_df = pd.DataFrame(exceptional_puzzles).merge(
    gold_exceptional_df,
    on="quizzes",
    how="left",
)

print("\nExceptional puzzles and their metrics")
print(
    exceptional_df.drop(columns=["quizzes", "solutions"])
    .to_string(index=False)
)

In [0]:
#Notes:
    #Puzzle 1725 required more attempts and backtracking than puzzle 5357, even though both began with 33 pre-filled numbers
    #This already reveals something interesting: the initial number of clues, on its own, does not fully account for the effort required to solve a puzzle
    #However, these are just two exceptional cases within a sample of 10,000 puzzles. We cannot draw general conclusions about difficulty based on them

In [0]:
#Evaluate the relationship between features and search effort
analysis_df = gold_sample_pd.reset_index(drop=True).copy()

analysis_df["search_nodes"] = backtracking_df["search_nodes"].to_numpy()
analysis_df["backtracks"] = backtracking_df["backtracks"].to_numpy()
analysis_df["max_search_depth"] = (
    backtracking_df["max_search_depth"].to_numpy()
)

columns_to_analyze = [
    "given_cells",
    "row_clue_std",
    "column_clue_std",
    "box_clue_std",
    "avg_candidate_count",
    "single_candidate_cells",
    "max_candidate_count",
    "search_nodes",
    "backtracks",
    "max_search_depth",
]

print(
    analysis_df[columns_to_analyze]
    .corr(method="spearman")
    .round(4)
    .to_string()
)

#Notes:
    #Spearman's correlation helps identify monotonic relationships without requiring them to be linear. Since nearly all backtrack values ​​are zero, the correlation should be interpreted with caution (~+1 -> perfect correlation, ~-1 -> perfect anti-correlation, ~0 -> no correlation)
    #Why use Spearman instead of Pearson?
        #Pearson measures the strength of a linear relationship between the values ​​of variables
        #Spearman assesses the relationship between their relative positions (ranks), allowing for the identification of monotonic relationships that are not linear, and is less sensitive to extreme values ​​in certain situations

In [0]:
#Notes:
    #given_cells × avg_candidate_count -> -0.532 -> Moderate negative association
    #avg_candidate_count × single_candidate_cells -> -0.636 -> Strong negative association
    #avg_candidate_count × search_nodes -> +0.532 -> Moderate positive association
    #given_cells × search_nodes	-> -0.999 -> Result that requires verification

In [0]:
#Associate solver metrics with their original puzzles
backtracking_df["quizzes"] = [
    record["quizzes"] for record in sample_df
]

#Retrieve Gold features for the same puzzles
sample_puzzles = backtracking_df["quizzes"].tolist()

gold_sample_pd = (
    spark.table("workspace.sudoku_gold.game_features")
    .filter(F.col("quizzes").isin(sample_puzzles))
    .select(
        "quizzes",
        "given_cells",
        "row_clue_std",
        "column_clue_std",
        "box_clue_std",
        "avg_candidate_count",
        "single_candidate_cells",
        "max_candidate_count",
    )
    .toPandas()
)

#Join by puzzle identity, not by row position
analysis_df = backtracking_df.merge(
    gold_sample_pd,
    on="quizzes",
    how="inner",
    validate="one_to_one",
)

columns_to_analyze = [
    "given_cells",
    "row_clue_std",
    "column_clue_std",
    "box_clue_std",
    "avg_candidate_count",
    "single_candidate_cells",
    "max_candidate_count",
    "search_nodes",
    "backtracks",
    "max_search_depth",
]

print(f"Matched puzzles: {len(analysis_df)}")

print(
    analysis_df[columns_to_analyze]
    .corr(method="spearman")
    .round(4)
    .to_string()
)

In [0]:
#Notes:
    #given_cells × search_nodes -> -0.9999 -> More clues, fewer search nodes
    #given_cells × max_search_depth	-> -1.0000 -> More clues, shallower search depth
    #search_nodes × max_search_depth -> +0.9999 -> More nodes, greater search depth
    #avg_candidate_count × search_nodes	-> +0.5319 -> More candidates, tendency toward more nodes
    #avg_candidate_count × single_candidate_cells -> -0.6356 -> More candidates, tendency toward fewer cells with a single candidate
    #given_cells × backtracks -> -0.0154 -> Virtually no monotonic association

In [0]:
#Insights:
    #Don't use `search_nodes` on its own as a difficulty label for the model. Instead, we need to evaluate a solver that accounts for more sophisticated solving techniques—such as naked pairs, pointing pairs, and other strategies. This way, we’ll have a better chance of generating a metric that shows real variation across the puzzles
    #Advanceds tests:
        #Naked Pairs -> Identify two cells within the same unit that share the exact same two candidates, and eliminate those candidates from all other cells in that unit
        #Pointing Pairs -> Identify when a candidate within a 3×3 block appears only in a single row or column of that block, and eliminate it from the remainder of that row or column
        #Re-evaluate the solver using a sample of 10,000 puzzles, measuring the number of puzzles solved, techniques employed, eliminations made, and backtracking attempts
        #Compare the distribution of these metrics against the current solver to determine whether we have achieved a target with sufficient variation for the model

## Advanced metrics

In [0]:
#Advanceds solvers:
    #Naked Pairs -> Identify two cells within the same unit that share the exact same two candidates, and eliminate those candidates from all other cells in that unit
    #Pointing Pairs -> Identify when a candidate within a 3×3 block appears only in a single row or column of that block, and eliminate it from the remainder of that row or column
    #Re-evaluate the solver using a sample of 10,000 puzzles, measuring the number of puzzles solved, techniques employed, eliminations made, and backtracking attempts
    #Compare the distribution of these metrics against the current solver to determine whether we have achieved a target with sufficient variation for the model

In [0]:
#Inspect the current solver function definitions
print(inspect.getsource(get_cell_candidates))
print(inspect.getsource(find_hidden_singles))

### Naked pair

In [0]:
#Load the first puzzle from the sample
test_quiz = sample_df[0]["quizzes"]

test_grid = [
    [int(value) for value in test_quiz[i:i + 9]]
    for i in range(0, 81, 9)
]

#Display the board and detected naked pairs
for row in test_grid:
    print(row)

print("\nNaked pair eliminations:")
print(find_naked_pairs(test_grid))

In [0]:
#Manual logical test
#Create a controlled Sudoku board for testing
test_grid = [
    [0, 0, 3, 4, 5, 6, 7, 8, 9],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
]

print("Candidates at (0, 0):", get_cell_candidates(test_grid, 0, 0))
print("Candidates at (0, 1):", get_cell_candidates(test_grid, 0, 1))
print("Naked pair eliminations:", find_naked_pairs(test_grid))

In [0]:
#Search for puzzles containing at least one naked pair elimination
puzzles_with_pairs = []

for row in sample_df:
    quiz = row["quizzes"]

    grid = [
        [int(value) for value in quiz[i:i + 9]]
        for i in range(0, 81, 9)
    ]

    eliminations = find_naked_pairs(grid)

    if eliminations:
        puzzles_with_pairs.append({
            "quizzes": quiz,
            "elimination_cells": len(eliminations),
            "eliminations": eliminations,
        })

        if len(puzzles_with_pairs) >= 5:
            break

print(f"Puzzles with naked pair eliminations: {len(puzzles_with_pairs)}")

for puzzle in puzzles_with_pairs:
    print(puzzle)

In [0]:
#Inspect naked pairs in the first puzzle
quiz = puzzles_with_pairs[0]["quizzes"]

grid = [
    [int(value) for value in quiz[i:i + 9]]
    for i in range(0, 81, 9)
]

units = []

#Rows
for row in range(9):
    units.append(("row", row, [(row, col) for col in range(9)]))

#Columns
for col in range(9):
    units.append(("column", col, [(row, col) for row in range(9)]))

#3x3 boxes
for box_row in range(0, 9, 3):
    for box_col in range(0, 9, 3):
        cells = [
            (row, col)
            for row in range(box_row, box_row + 3)
            for col in range(box_col, box_col + 3)
        ]
        units.append(("box", (box_row, box_col), cells))

for unit_type, unit_id, cells in units:
    candidate_cells = {}

    for row, col in cells:
        candidates = get_cell_candidates(grid, row, col)

        if len(candidates) == 2:
            pair = tuple(candidates)
            candidate_cells.setdefault(pair, []).append((row, col))

    for pair, pair_cells in candidate_cells.items():
        if len(pair_cells) == 2:
            print(
                f"Unit: {unit_type} {unit_id} | "
                f"Pair: {pair} | "
                f"Cells: {pair_cells}"
            )

In [0]:
#Notes:
    #This result revealed an important point regarding our implementation: the first Sudoku contains two valid Naked Pairs, but they belong to different units

In [0]:
#Validate eliminations for each naked pair independently
for unit_type, unit_id, cells in units:
    candidate_cells = {}

    for row, col in cells:
        candidates = get_cell_candidates(grid, row, col)

        if len(candidates) == 2:
            pair = tuple(candidates)
            candidate_cells.setdefault(pair, []).append((row, col))

    for pair, pair_cells in candidate_cells.items():
        if len(pair_cells) != 2:
            continue

        pair_numbers = set(pair)

        for row, col in cells:
            if (row, col) in pair_cells or grid[row][col] != 0:
                continue

            candidates = set(get_cell_candidates(grid, row, col))
            removable = candidates & pair_numbers

            if removable:
                print(
                    f"Unit: {unit_type} {unit_id} | "
                    f"Pair: {pair} | "
                    f"Pair cells: {pair_cells} | "
                    f"Target: {(row, col)} | "
                    f"Remove: {sorted(removable)}"
                )

In [0]:
#Notes:
    #In row 1, the numbers 1 and 9 must occupy cells (1, 0) and (1, 4). Therefore, they cannot appear as candidates in the other cells of that row. The same logic applies to the top-left block, considering cells (0, 1) and (1, 0)

In [0]:
#Insights:
    #Candidate map: 
        #The idea is to create a function that initializes the candidates for each empty cell
        #Eliminations made via "Naked Pairs" will be stored within this structure, rather than being lost when we recalculate the candidates based on the board state
    #The idea is for the solver to apply the techniques in this order:
        #Naked Singles: fill in cells that have only one candidate
        #Hidden Singles: fill in cells where a number can only occupy one position within a unit
        #Naked Pairs: eliminate candidates from other cells within the same unit
        #Repeat the cycle until the Sudoku is solved or no further progress can be made

### Combine Solver

In [0]:
#Validate the candidate map logic
quiz = puzzles_with_pairs[0]["quizzes"]

test_grid = [
    [int(value) for value in quiz[i:i + 9]]
    for i in range(0, 81, 9)
]

candidate_map = initialize_candidates(test_grid)

print("Empty cells:", len(candidate_map))
print("Candidates at (1, 3):", candidate_map.get((1, 3)))
print("Candidates at (1, 6):", candidate_map.get((1, 6)))
print("Candidates at (0, 2):", candidate_map.get((0, 2)))


In [0]:
#Validate the elimination logic
candidate_map = initialize_candidates(test_grid)

before = {
    cell: candidate_map[cell].copy()
    for cell in [(1, 3), (1, 6), (0, 2)]
}

eliminations_count = apply_naked_pairs(test_grid, candidate_map)

print("Eliminated candidates:", eliminations_count)

for cell in [(1, 3), (1, 6), (0, 2)]:
    print(
        f"Cell {cell}: "
        f"before={sorted(before[cell])}, "
        f"after={sorted(candidate_map[cell])}"
    )


In [0]:
#validate the propagation logic
quiz = puzzles_with_pairs[0]["quizzes"]

test_grid = [
    [int(value) for value in quiz[i:i + 9]]
    for i in range(0, 81, 9)
]

reference_solution = [
    int(value)
    for value in sample_df[0]["solutions"]
]

candidate_map = initialize_candidates(test_grid)
result = propagate_sudoku(test_grid, candidate_map)

solved_values = [
    value
    for row in test_grid
    for value in row
]

print("Solved:", result["solved"])
print("Contradiction:", result["contradiction"])
print("Naked Single steps:", result["naked_single_steps"])
print("Hidden Single steps:", result["hidden_single_steps"])
print("Naked Pair eliminations:", result["naked_pair_eliminations"])
print("Solution matches reference:", solved_values == reference_solution)
print("Remaining empty cells:", sum(value == 0 for value in solved_values))


In [0]:
#Insights:
    #The solver filled in the 47 empty cells, but this does not guarantee that the solution is correct
    #The comparison with the reference solution returned False. Since we are using `puzzles_with_pairs[0]`, we must ensure that the reference solution corresponds exactly to this specific Sudoku puzzle
    #Furthermore, "Naked Pair eliminations: 0" does not necessarily mean that the technique failed; the solver may have solved the board using "Naked Singles" before the need to apply "Naked Pairs" arose

In [0]:
#Correct the comparison with the reference
quiz = puzzles_with_pairs[0]["quizzes"]

reference_row = next(
    row for row in sample_df
    if row["quizzes"] == quiz
)

reference_solution = reference_row["solutions"]

solved_values = "".join(
    str(value)
    for row in test_grid
    for value in row
)

print("Same puzzle:", quiz == reference_row["quizzes"])
print("Solution matches reference:", solved_values == reference_solution)
print("Reference length:", len(reference_solution))
print("Calculated solution length:", len(solved_values))
print("Reference solution:", reference_solution)
print("Calculated solution:", solved_values)


In [0]:
#Notes:
    #Great! We have confirmed that the solution is correct and matches the dataset reference exactly. The previous "False" result was simply due to a comparison against the solution for a different Sudoku puzzle
    #The solver solved the board using 47 Naked Singles
    #No Hidden Singles were required
    #No Naked Pairs were needed, as the preceding techniques had already solved the Sudoku before reaching that stage

In [0]:
#Finding a Sudoku that truly requires Naked Pairs
pair_candidates = []

for row in sample_df:
    quiz = row["quizzes"]

    grid = [
        [int(value) for value in quiz[i:i + 9]]
        for i in range(0, 81, 9)
    ]

    candidates = initialize_candidates(grid)
    result = propagate_sudoku(grid, candidates)

    if not result["solved"] and not result["contradiction"]:
        pair_candidates.append({
            "quizzes": quiz,
            "remaining_empty_cells": sum(
                value == 0
                for board_row in grid
                for value in board_row
            ),
            "naked_pair_eliminations": result["naked_pair_eliminations"],
        })

        if len(pair_candidates) >= 5:
            break

print("Puzzles not solved by propagation:", len(pair_candidates))

for puzzle in pair_candidates:
    print(puzzle)


In [0]:
#Notes:
    #The search returned zero unsolved Sudoku puzzles because all the games in `sample_df` were solved by the implemented propagation method. This is consistent with previous tests, in which "Naked Singles" and "Hidden Singles" had already solved virtually all cases
    #There is, however, a technical detail: our `propagate_sudoku()` function may fill all cells using "Naked Singles" before applying "Naked Pairs." Therefore, this result does not prove that "Naked Pairs" is unnecessary in general; it only shows that it was not required for the games that were tested

In [0]:
#Measuring the use of Naked Pairs technique
pair_usage = []
total_eliminations = 0
puzzles_with_eliminations = 0

for row in sample_df:
    quiz = row["quizzes"]

    grid = [
        [int(value) for value in quiz[i:i + 9]]
        for i in range(0, 81, 9)
    ]

    candidates = initialize_candidates(grid)
    result = propagate_sudoku(grid, candidates)

    eliminations = result["naked_pair_eliminations"]
    total_eliminations += eliminations

    if eliminations > 0:
        puzzles_with_eliminations += 1

    pair_usage.append(eliminations)

print("Puzzles analyzed:", len(sample_df))
print("Puzzles with Naked Pair eliminations:", puzzles_with_eliminations)
print("Total candidate eliminations:", total_eliminations)
print(
    "Average eliminations per puzzle:",
    round(total_eliminations / len(sample_df), 4)
)


In [0]:
#Insights:
    #The problem is that `propagate_sudoku()` fills all cells with "Naked Singles" before recalculating the board state. This can prevent the "Naked Pairs" technique from ever being executed, even when relevant candidates still exist. Furthermore, since the candidate map is rebuilt after each round of filling, we must ensure that eliminations are correctly preserved between these rounds

In [0]:
#Isolate the test: verify whether `apply_naked_pairs()` finds eliminations on a board before applying Singles
pair_test_results = []

for row in sample_df:
    quiz = row["quizzes"]

    grid = [
        [int(value) for value in quiz[i:i + 9]]
        for i in range(0, 81, 9)
    ]

    candidates = initialize_candidates(grid)
    eliminations = apply_naked_pairs(grid, candidates)

    if eliminations > 0:
        pair_test_results.append({
            "quizzes": quiz,
            "eliminations": eliminations,
        })

        if len(pair_test_results) >= 10:
            break

print("Puzzles with initial Naked Pair eliminations:", len(pair_test_results))

for result in pair_test_results:
    print(result)


In [0]:
#Notes:
    #This result indicates that the issue lies in the integration with the propagation phase, rather than in the initial detection of the technique
    #Let’s address this in a controlled manner: first, we will compare the propagation of "Singles" with the application of "Naked Pairs" in one of the puzzles that we know contains eliminations

In [0]:
#Compare before and after the Naked Pairs technique
quiz = pair_test_results[0]["quizzes"]

grid = [
    [int(value) for value in quiz[i:i + 9]]
    for i in range(0, 81, 9)
]

candidates_before = initialize_candidates(grid)

initial_eliminations = apply_naked_pairs(
    grid,
    candidates_before,
)

remaining_candidates = sum(
    len(values)
    for values in candidates_before.values()
)

print("Initial Naked Pair eliminations:", initial_eliminations)
print("Empty cells:", len(candidates_before))
print("Remaining candidates after elimination:", remaining_candidates)
print("Cells with one candidate:", sum(
    len(values) == 1
    for values in candidates_before.values()
))
print("Cells with zero candidates:", sum(
    len(values) == 0
    for values in candidates_before.values()
))


In [0]:
#Insights:
    #The most interesting detail is that the elimination of candidates can create new opportunities for "Naked Singles" Therefore, the solver must alternate between these techniques until the state stabilizes
    #There is also a methodological point to consider: we must avoid re-counting an elimination that has already been performed, and we cannot rebuild the map from the board in every round, as doing so would erase the eliminations made via "Naked Pairs"
    #We need to correct propagate_sudoku function -> Create new version of propagate_sudoku function


In [0]:
#New propagate function test + new candidates map
quiz = pair_test_results[0]["quizzes"]

test_grid = [
    [int(value) for value in quiz[i:i + 9]]
    for i in range(0, 81, 9)
]

candidate_map = new_initialize_candidates(test_grid)
result = new_propagate_function(test_grid, candidate_map)

print("Solved:", result["solved"])
print("Contradiction:", result["contradiction"])
print("Naked Single steps:", result["naked_single_steps"])
print("Hidden Single steps:", result["hidden_single_steps"])
print("Naked Pair eliminations:", result["naked_pair_eliminations"])
print("Remaining empty cells:", sum(
    value == 0 for row in test_grid for value in row
))


In [0]:
#Compare the order of the techniques
quiz = pair_test_results[0]["quizzes"]

test_grid = [
    [int(value) for value in quiz[i:i + 9]]
    for i in range(0, 81, 9)
]

candidate_map = new_initialize_candidates(test_grid)

initial_eliminations = apply_naked_pairs(
    test_grid,
    candidate_map
)

print("Initial Naked Pair eliminations:", initial_eliminations)
print(
    "Cells with one candidate after Naked Pairs:",
    sum(len(values) == 1 for values in candidate_map.values())
)
print("Remaining empty cells:", len(candidate_map))


In [0]:
#Controlled test: Test for Naked Pairs before Singles
quiz = pair_test_results[0]["quizzes"]

test_grid = [
    [int(value) for value in quiz[i:i + 9]]
    for i in range(0, 81, 9)
]

candidate_map = new_initialize_candidates(test_grid)

initial_pair_eliminations = apply_naked_pairs(
    test_grid,
    candidate_map
)

result = new_propagate_function(test_grid, candidate_map)

print("Initial Naked Pair eliminations:", initial_pair_eliminations)
print("Solved:", result["solved"])
print("Contradiction:", result["contradiction"])
print("Naked Single steps:", result["naked_single_steps"])
print("Hidden Single steps:", result["hidden_single_steps"])
print("Naked Pair eliminations during propagation:", result["naked_pair_eliminations"])
print("Remaining empty cells:", sum(
    value == 0 for row in test_grid for value in row
))


In [0]:
#Notes:
    #We applied Naked Pairs before propagation and eliminated 4 candidates
    #Propagation resolved the 47 cells using Naked Singles
    #No further eliminations were required during propagation

In [0]:
#Compare the solution with the reference
reference_row = next(
    row for row in sample_df
    if row["quizzes"] == quiz
)

reference_solution = reference_row["solutions"]

solved_values = "".join(
    str(value)
    for row in test_grid
    for value in row
)

total_pair_eliminations = (
    initial_pair_eliminations
    + result["naked_pair_eliminations"]
)

print("Solution matches reference:", solved_values == reference_solution)
print("Initial Naked Pair eliminations:", initial_pair_eliminations)
print("Additional Naked Pair eliminations:", result["naked_pair_eliminations"])
print("Total Naked Pair eliminations:", total_pair_eliminations)
print("Total solved cells:", sum(
    value != 0 for row in test_grid for value in row
))

In [0]:
#Notes:
    #The test confirmed that the eliminations of Naked Pairs did not alter the final solution and that the board was completely solved

In [0]:
#Insights:
    #We need to integrate the application of Naked Pairs into the `new_propagate_function` function, rather than applying it before propagation. We should also adjust the order of the techniques so that eliminations can influence the identification of new cells with unique candidates
    #The first adjustment will be to have `new_propagate_function()` apply Naked Pairs before searching for Naked Singles and Hidden Singles. This way, the eliminations can result in new cells with only a single candidate, and the function will account for these eliminations during the propagation process itself

### New Combine Solver + Auxiliary Hidden Steps 

In [0]:
#Manual test for new propagate function with auxiliary hidden steps
quiz = (
    "600050000073008020854027000201700530400069007080000900"
    "027301084060540009300000001"
)

test_grid = [
    [int(quiz[row * 9 + col]) for col in range(9)]
    for row in range(9)
]

candidate_map = new_initialize_candidates(test_grid)

result = new_propagate_function(test_grid, candidate_map)

reference_row = next(
    row for row in sample_df
    if row["quizzes"] == quiz
)

reference_solution = reference_row["solutions"]

solved_values = "".join(
    str(value)
    for row in test_grid
    for value in row
)

print("Solved:", result["solved"])
print("Contradiction:", result["contradiction"])
print("Solution matches reference:", solved_values == reference_solution)
print("Naked Single steps:", result["naked_single_steps"])
print("Hidden Single steps:", result["hidden_single_steps"])
print("Naked Pair eliminations:", result["naked_pair_eliminations"])
print("Remaining empty cells:", sum(
    value == 0 for row in test_grid for value in row
))

In [0]:
#Test for one thousand puzzles
test_results = []

for row in sample_df[:1000]:
    quiz = row["quizzes"]
    reference_solution = row["solutions"]

    grid = [
        [int(quiz[r * 9 + c]) for c in range(9)]
        for r in range(9)
    ]

    candidates = new_initialize_candidates(grid)
    result = new_propagate_function(grid, candidates)

    solved_values = "".join(
        str(value)
        for board_row in grid
        for value in board_row
    )

    test_results.append({
        "solved": result["solved"],
        "contradiction": result["contradiction"],
        "matches_reference": solved_values == reference_solution,
        "naked_single_steps": result["naked_single_steps"],
        "hidden_single_steps": result["hidden_single_steps"],
        "naked_pair_eliminations": result["naked_pair_eliminations"],
        "remaining_empty_cells": sum(
            value == 0
            for board_row in grid
            for value in board_row
        ),
    })

total = len(test_results)

print("Puzzles tested:", total)
print("Solved:", sum(r["solved"] for r in test_results))
print("Contradictions:", sum(r["contradiction"] for r in test_results))
print("Solutions matching reference:", sum(
    r["matches_reference"] for r in test_results
))
print("Total Naked Pair eliminations:", sum(
    r["naked_pair_eliminations"] for r in test_results
))
print("Puzzles using Naked Pairs:", sum(
    r["naked_pair_eliminations"] > 0 for r in test_results
))
print("Puzzles with remaining empty cells:", sum(
    r["remaining_empty_cells"] > 0 for r in test_results
))


In [0]:
#Analyze the distribution of effort
results_df = pd.DataFrame(test_results)

metrics = [
    "naked_single_steps",
    "hidden_single_steps",
    "naked_pair_eliminations",
]

print("Metric summary:")
display(results_df[metrics].describe().T)

print("\nNaked Pair elimination distribution:")
display(
    results_df["naked_pair_eliminations"]
    .value_counts()
    .sort_index()
    .rename_axis("eliminations")
    .reset_index(name="puzzle_count")
)

print("\nNaked Pair elimination percentiles:")
display(
    results_df["naked_pair_eliminations"]
    .quantile([0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00])
    .rename("naked_pair_eliminations")
    .to_frame()
)


In [0]:
#Validate the consistency of the solver
validation_df = pd.DataFrame(test_results)

print("Validation summary")
print("------------------")
print("Total puzzles:", len(validation_df))
print("All solved:", validation_df["solved"].all())
print("Any contradictions:", validation_df["contradiction"].any())
print("All solutions match reference:", validation_df["matches_reference"].all())
print("All grids complete:", (validation_df["remaining_empty_cells"] == 0).all())

print("\nNaked Pair elimination statistics")
print("---------------------------------")
print(validation_df["naked_pair_eliminations"].describe())

print("\nPuzzles with zero Naked Pair eliminations:",
      (validation_df["naked_pair_eliminations"] == 0).sum())

print("Puzzles with more than 20 eliminations:",
      (validation_df["naked_pair_eliminations"] > 20).sum())


In [0]:
#Notes:
    #The most interesting piece of data is that 97 puzzles featured more than 20 "Naked Pair" eliminations. We therefore have a metric with sufficient variation to investigate as part of the algorithmic effort

In [0]:
#Test for ten thousand random puzzles
random.seed(42) #42 is the answer to life, the universe and everything

test_sample = random.sample(sample_df, min(10_000, len(sample_df)))
test_results_10k = []

for row in test_sample:
    quiz = row["quizzes"]
    reference_solution = row["solutions"]

    grid = [
        [int(quiz[r * 9 + c]) for c in range(9)]
        for r in range(9)
    ]

    candidates = new_initialize_candidates(grid)
    result = new_propagate_function(grid, candidates)

    solved_values = "".join(
        str(value)
        for board_row in grid
        for value in board_row
    )

    test_results_10k.append({
        "solved": result["solved"],
        "contradiction": result["contradiction"],
        "matches_reference": solved_values == reference_solution,
        "naked_single_steps": result["naked_single_steps"],
        "hidden_single_steps": result["hidden_single_steps"],
        "naked_pair_eliminations": result["naked_pair_eliminations"],
        "remaining_empty_cells": sum(
            value == 0
            for board_row in grid
            for value in board_row
        ),
    })

results_10k_df = pd.DataFrame(test_results_10k)

print("Puzzles tested:", len(results_10k_df))
print("Solved:", results_10k_df["solved"].sum())
print("Contradictions:", results_10k_df["contradiction"].sum())
print("Solutions matching reference:", results_10k_df["matches_reference"].sum())
print("Incomplete grids:", (results_10k_df["remaining_empty_cells"] > 0).sum())

print("\nEffort metrics:")
display(
    results_10k_df[
        [
            "naked_single_steps",
            "hidden_single_steps",
            "naked_pair_eliminations",
        ]
    ].describe().T
)


In [0]:
#Notes:
    #The main finding is that Naked Pairs adds variation, whereas Naked Singles remains highly concentrated. This reinforces the idea of ​​investigating the number of eliminations as one of the metrics for algorithmic effort

In [0]:

#Compare the effort of the solvers
#Perform backtracking on the same sample of 10,000 puzzles that we used in the propagation test, and calculate the Spearman correlation between: naked_pair_eliminations, naked_single_steps, hidden_single_steps, search_nodes, backtracks and max_search_depth

#Associate each propagation result with its original Sudoku.
propagation_df = pd.DataFrame([
    {
        "quizzes": sample_row["quizzes"],
        "naked_pair_eliminations": result["naked_pair_eliminations"],
        "naked_single_steps": result["naked_single_steps"],
        "hidden_single_steps": result["hidden_single_steps"],
    }
    for sample_row, result in zip(test_sample, test_results_10k)
])

#Match both solvers using the Sudoku puzzle as the key.
comparison_df = propagation_df.merge(
    backtracking_df,
    on="quizzes",
    validate="one_to_one",
)

print("Puzzles matched:", len(comparison_df))

correlation_columns = [
    "naked_pair_eliminations",
    "naked_single_steps",
    "search_nodes",
    "backtracks",
    "max_search_depth",
]

print("\nSpearman correlation matrix:")
display(
    comparison_df[correlation_columns]
    .corr(method="spearman")
    .round(4)
)

print("\nBacktracking metrics summary:")
display(
    comparison_df[
        ["search_nodes", "backtracks", "max_search_depth"]
    ].describe().T.round(4)
)


In [0]:
#Notes:
    #Spearman:
        #Naked Pairs × Search Nodes -> 0.2024 -> Weak positive correlation
        #Naked Pairs × Max Search Depth -> 0.2023 -> Weak positive correlation
        #Naked Pairs × Naked Singles -> 0.2023 -> Weak positive correlation
        #Naked Singles × Search Nodes -> 0.9999	-> Almost perfect correlation
        #Backtracks × Search Nodes -> 0.0261 -> Virtually no correlation
    #Backtracking metrics:
        #search_nodes -> The average is approximately 48 nodes visited. Since the puzzles contain between 44 and 52 empty cells, this metric is strongly related to the number of cells that need to be filled
        #backtracks -> The median is zero, and the third quartile is also zero. This means that at least 75% of the puzzles did not require any backtracking
        #max_search_depth -> The average is approximately 47 levels, which is very close to the average number of empty cells
    #Important:
        #Naked Pairs -> This adds some information, but on its own, it does not seem sufficient to represent the complexity of a Sudoku puzzle
        #Naked Singles -> The near-perfect correlation with `search_nodes` indicates that both metrics essentially reflect the number of empty cells and the progress made by the solver
        #Backtracks -> Since almost none of the puzzles required backtracking, this metric has little capacity to differentiate between the cases in this sample

## Solver decision

In [0]:
#Three steps:
    #1. Define the complexity target based on available metrics
    #2. Create 'Easy', 'Medium', and 'Hard' classes with documented criteria
    #3. Save the targets to the Gold table in Databricks, preparing the foundation for training with Spark MLlib

### Analyze a sample of ten thousand puzzles

In [0]:
#Analyze the distribution of backtracking metrics to define a solving_complexity
print("comparison_columns_df:")
print(comparison_df.columns)

print("\nBacktracks:")
display(
    comparison_df["backtracks"]
    .value_counts()
    .sort_index()
    .to_frame("puzzle_count")
)

In [0]:
#Examine the distribution of eliminations resulting from Naked Pairs and the distribution of non-zero backtrack values
print("Naked Pairs Elimination Summary:")
display(
    comparison_df["naked_pair_eliminations"]
    .describe()
    .to_frame("naked_pair_eliminations")
    .round(4)
)

print("Puzzles Requiring Backtracking:")
display(
    comparison_df.loc[
        comparison_df["backtracks"] > 0,
        ["quizzes", "naked_pair_eliminations", "backtracks"]
    ]
)

In [0]:
#Insights:
    #Initial target:
        #solving_complexity -> The number of candidate eliminations via Naked Pairs, plus a penalty applied when backtracking is required
        #difficulty_label -> Relative categories (Easy, Medium, Hard) derived from the tertiles of the target distribution—these do not represent difficulty as validated by human solvers (1/3 and 2/3)

In [0]:
#Start with `naked_pair_eliminations` as an experimental target and document this limitation
target_df = (
    spark.createDataFrame(
        comparison_df[
            ["quizzes", "naked_pair_eliminations", "backtracks"]
        ]
    )
    .withColumn(
        "solving_complexity",
        F.col("naked_pair_eliminations").cast("double"),
    )
)

print("Experimental Target Summary:")
display(
    target_df.select("solving_complexity")
    .describe()
)

print("Experimental Target Distribution:")
display(
    target_df.groupBy("solving_complexity")
    .count()
    .orderBy("solving_complexity")
)

In [0]:
#Calculate the 33.3rd and 66.7th percentiles of the experimental target
#These thresholds divide the target distribution into three approximate groups
quantiles = target_df.approxQuantile(
    "solving_complexity",
    [1 / 3, 2 / 3],
    0.001,  #Relative error allowed in the approximate quantile calculation
)

#Store the percentile thresholds for the classification rules
lower_threshold, upper_threshold = quantiles

#Assign a difficulty label based on the target's percentile thresholds:
    #Easy -> solving_complexity is less than or equal to the lower threshold
    #Medium -> solving_complexity is above the lower threshold and less than or equal to the upper threshold
    #Hard -> solving_complexity is above the upper threshold

target_df = target_df.withColumn(
    "difficulty_label",
    F.when(
        F.col("solving_complexity") <= lower_threshold,
        "Easy",
    )
    .when(
        F.col("solving_complexity") <= upper_threshold,
        "Medium",
    )
    .otherwise("Hard"),
)

#Display the thresholds used to assign the difficulty labels
print("Difficulty Classification Thresholds:")
print(f"Easy upper threshold: {lower_threshold}")
print(f"Medium upper threshold: {upper_threshold}")
print(f"Hard lower threshold: {upper_threshold}")

#Count the puzzles assigned to each difficulty class
#Use a custom ordering to display the classes as Easy, Medium, and Hard.
print("Difficulty Class Distribution:")
display(
    target_df.groupBy("difficulty_label")
    .count()
    .orderBy(
        F.when(F.col("difficulty_label") == "Easy", 1)
        .when(F.col("difficulty_label") == "Medium", 2)
        .otherwise(3)
    )
)


In [0]:
#Notes:
    #Easy -> solving_complexity <= 9 -> 3385 (33.85%)
    #Medium -> 9 < solving_complexity <= 15 -> 3779 (37.79%)
    #Hard -> solving_complexity > 15 -> 2836 (28.36%)

### Applying to the gold table

In [0]:
#Load the Gold feature table
gold_df = spark.table("workspace.sudoku_gold.game_features")

#Display the table schema
print("Gold Feature Table Schema:")
gold_df.printSchema()

#Check the total number of records
print("Gold Feature Table Record Count:")
print(gold_df.count())

#Display a sample of the available features
display(gold_df.limit(5))
